Cell 1: Setup & Configuration
This cell initializes the project, checks for available hardware, and sets up a global configuration dictionary for all key parameters.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import os
import cv2
import re
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
#from torch.cuda.amp import GradScaler
from albumentations.pytorch import ToTensorV2
import albumentations as A
from tqdm import tqdm
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from transformers import SegformerForSemanticSegmentation
import matplotlib.colors as mcolors
import math
from contextlib import nullcontext



# Hardware & Configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Running on: {DEVICE}")

CONFIG = {
    # Input
    'img_height': 480,
    'img_width': 640,
    
    # Heads
    'num_seg_classes': 10,
    'num_det_classes': 91,
    
    # Stereo Geometry
    'max_disp_pixel': 192,
    'backbone_stride': 4,
    'internal_disp_steps': 48, # 192px / stride 4 = 48
    
    # Training
    'batch_size': 4,
    'ACCUMULATION_STEPS': 24,
    'lr': 2e-4,
    'num_epochs': 25,
    'num_workers': 4,
    'save_dir': "./checkpoints"
}
os.makedirs(CONFIG['save_dir'], exist_ok=True)

print("✅ Setup complete. Configuration loaded.")


🚀 Running on: cuda
✅ Setup complete. Configuration loaded.


Cell 2: Fused Model Architecture
This cell contains the complete, final architecture for the FusedHexapodModel, including the shared MobileNetV3 backbone and all three specialized heads (Stereo, Segmentation, and Detection).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import math

# --- Constants for Fixed Resolutions (Hailo Requirement) ---
# Input: 640x480
RES_S8 = (60, 80)    # Stride 8
RES_S4 = (120, 160)  # Stride 4
RES_S1 = (480, 640)  # Full Res

# --- Helper Blocks ---
class ConvBnReLU(nn.Module):
    """ Standard Conv-BN-ReLU block """
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, dilation=1):
        super().__init__()
        padding = ((kernel_size - 1) * dilation) // 2
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class ResBlock(nn.Module):
    """ Lightweight Residual Block for Refinement """
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBnReLU(channels, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.bn2(self.conv2(self.conv1(x))))

class ConvBR(nn.Module):
    """Helper: Conv -> BN -> ReLU for Heads"""
    def __init__(self, c_in, c_out, kernel=3, stride=1):
        super().__init__()
        padding = (kernel - 1) // 2
        self.conv = nn.Conv2d(c_in, c_out, kernel_size=kernel, stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(c_out)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

# --- Stereo Components ---
class CoarseCostVolume(nn.Module):
    """
    Hailo-Optimized Cost Volume.
    Uses 'Slice' operations (generated via loop) which map to 
    zero-cost memory pointers on the NPU.
    """
    def __init__(self, max_disp, in_channels):
        super().__init__()
        self.max_disp = max_disp
        # 1. Compress Inputs (32ch -> 16ch) to save bandwidth
        self.compress = nn.Conv2d(in_channels, 16, 1, bias=False)
        
        # 2. Grouped Conv for Summation (Replaces Mean)
        # Input: [B, 16, H, W] -> Output: [B, 1, H, W]
        self.reduce = nn.Conv2d(16, 1, 1, bias=False)
        with torch.no_grad():
            self.reduce.weight.fill_(1.0 / 16)
            self.reduce.weight.requires_grad = False

    def forward(self, left_feat, right_feat):
        # [B, 16, H, W]
        l = self.compress(left_feat)
        r = self.compress(right_feat)
        
        cost_list = []
        
        # --- The "Compiler-Friendly" Loop ---
        # This loop vanishes at compile time, creating 24 parallel branches.
        for d in range(self.max_disp):
            if d == 0:
                sim = self.reduce(l * r)
                cost_list.append(sim)
            else:
                # Slice Right: r[:, :, :, :-d]
                r_shift = r[:, :, :, : -d]
                # Slice Left: l[:, :, :, d:]
                l_match = l[:, :, :, d :]
                
                # Multiply & Reduce
                product = l_match * r_shift
                sim = self.reduce(product)
                
                # Pad back to original size [B, 1, H, W]
                # (Left, Right, Top, Bottom)
                sim_padded = F.pad(sim, (d, 0, 0, 0))
                cost_list.append(sim_padded)
            
        # Concatenate -> [B, 24, H, W]
        return torch.cat(cost_list, dim=1)

class RefinementStage(nn.Module):
    """
    Hierarchical Refinement with Fixed Target Resolution.
    Upsample Disp -> Concat Guidance -> Predict Residual.
    """
    def __init__(self, guidance_channels, target_res):
        super().__init__()
        self.target_res = target_res # Tuple (H, W)
        
        self.net = nn.Sequential(
            ConvBnReLU(guidance_channels + 1, 32),
            ResBlock(32),
            ResBlock(32),
            ResBlock(32),
            nn.Conv2d(32, 1, 3, 1, 1, bias=True) # Output: Residual
        )
        # Zero initialization for stability
        nn.init.constant_(self.net[-1].weight, 0)
        nn.init.constant_(self.net[-1].bias, 0)

    def forward(self, disparity_low, guidance_features):
        # 1. Fixed Upsample (Static Shape for Hailo)
        disparity_up = F.interpolate(disparity_low, size=self.target_res, mode='bilinear', align_corners=False)
        
        # 2. Scale disparity (x2 for S8->S4, x4 for S4->S1)
        # Note: We calculate scale based on width ratio dynamically or fixed.
        # Here we assume a 2x step usually, but S4->S1 is 4x step.
        scale_factor = float(self.target_res[1]) / float(disparity_low.shape[-1])
        disparity_up = disparity_up * scale_factor
        
        # 3. Concat & Residual
        x = torch.cat([disparity_up, guidance_features], dim=1)
        residual = self.net(x)
        
        return F.relu(disparity_up + residual)

# --- Segmentation & Detection Heads ---
class LRASPPHead(nn.Module):
    def __init__(self, low_ch, high_ch, num_classes):
        super().__init__()
        self.cbr_high = nn.Sequential(
            nn.Conv2d(high_ch, 128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.scale_high = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(high_ch, 128, 1, bias=False), nn.Sigmoid()
        )
        self.low_classifier = nn.Conv2d(low_ch, num_classes, 1)
        self.high_classifier = nn.Conv2d(128, num_classes, 1)

    def forward(self, x_low, x_high):
        out = self.cbr_high(x_high) * self.scale_high(x_high)
        out = F.interpolate(out, size=x_low.shape[-2:], mode='bilinear', align_corners=False)
        return self.low_classifier(x_low) + self.high_classifier(out)

class DecoupledHead(nn.Module):
    def __init__(self, ch_in, num_classes, width=256):
        super().__init__()
        self.stem = ConvBR(ch_in, width, kernel=1, stride=1)
        self.reg_conv = nn.Sequential(ConvBR(width, width), ConvBR(width, width))
        self.reg_pred = nn.Conv2d(width, 4, kernel_size=1) 
        self.cls_conv = nn.Sequential(ConvBR(width, width), ConvBR(width, width))
        self.cls_pred = nn.Conv2d(width, num_classes, kernel_size=1)
        self.obj_pred = nn.Conv2d(width, 1, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)
        
        # Regressions-Pfad (Lage + Existenz)
        x_reg = self.reg_conv(x)
        reg_out = self.reg_pred(x_reg)
        obj_out = self.obj_pred(x_reg) # <- Von x_reg ableiten
        
        # Klassifizierungs-Pfad (Semantik)
        x_cls = self.cls_conv(x)
        cls_out = self.cls_pred(x_cls)
        
        return torch.cat([reg_out, obj_out, cls_out], dim=1)

class YOLOHead(nn.Module):
    def __init__(self, ch_dims, num_classes):
        super().__init__()
        self.head_s8  = DecoupledHead(ch_dims[1], num_classes)
        self.head_s16 = DecoupledHead(ch_dims[2], num_classes)
        self.head_s32 = DecoupledHead(ch_dims[3], num_classes)

    def forward(self, x_s8, x_s16, x_s32):
        return [self.head_s8(x_s8), self.head_s16(x_s16), self.head_s32(x_s32)]


# --- NEUER KOMBINIERTER HEAD ---
class HierarchicalStereoHead(nn.Module):
    def __init__(self, ch_s8, ch_s4, max_disp_s8, res_s4, res_s1):
        super().__init__()
        self.max_disp_s8 = max_disp_s8
        
        # 1. Feature Reducers
        self.reduce_s8 = nn.Conv2d(ch_s8, 32, 1, bias=False)
        self.reduce_s4 = nn.Conv2d(ch_s4, 32, 1, bias=False)
        
        # 2. Coarse Matching (Stride 8)
        self.stereo_coarse = CoarseCostVolume(max_disp=self.max_disp_s8, in_channels=32)
        
        # 3. Refinement Stages
        # FIX: Hier übergeben wir jetzt die echten Auflösungs-Tuples (H, W)
        self.stereo_refine_s4 = RefinementStage(guidance_channels=32, target_res=res_s4) 
        self.stereo_refine_s1 = RefinementStage(guidance_channels=1,  target_res=res_s1) 
        
        # 4. Disparity Register
        self.register_buffer('disp_reg', torch.arange(self.max_disp_s8, dtype=torch.float32).view(1, -1, 1, 1))

    def forward(self, l_s8, r_s8, l_s4, l_img_raw):
        # A. Feature Reduction
        feat_l_s8 = self.reduce_s8(l_s8)
        feat_r_s8 = self.reduce_s8(r_s8)
        feat_l_s4 = self.reduce_s4(l_s4)

        # B. Coarse Matching
        vol_s8 = self.stereo_coarse(feat_l_s8, feat_r_s8)
        prob_s8 = F.softmax(vol_s8, dim=1)
        disp_s8 = torch.sum(prob_s8 * self.disp_reg, dim=1, keepdim=True)
        
        # C. Refinement Hierarchy
        disp_s4 = self.stereo_refine_s4(disp_s8, feat_l_s4)
        final_disp = self.stereo_refine_s1(disp_s4, l_img_raw)
        
        return final_disp


# --- 2. KORRIGIERTES HAUPTMODELL ---
class FusedHexapodModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        
        # Auflösungen berechnen (Wichtig für Interpolation!)
        H, W = config['img_height'], config['img_width']
        res_s1 = (H, W)         # z.B. (480, 640)
        res_s4 = (H // 4, W // 4) # z.B. (120, 160)
        
        # Backbone
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True, features_only=True, out_indices=(1, 2, 3, 4))
        bb_ch = self.backbone.feature_info.channels() 
        
        # Stereo Setup
        self.max_disp_s8 = 24 
        
        # FIX: Übergabe der Auflösungen an den Head
        self.stereo_head = HierarchicalStereoHead(
            ch_s8=bb_ch[1],       
            ch_s4=bb_ch[0],       
            max_disp_s8=self.max_disp_s8,
            res_s4=res_s4,  # <--- NEU
            res_s1=res_s1   # <--- NEU
        )

        # Other Heads
        self.seg_head = LRASPPHead(low_ch=bb_ch[0], high_ch=bb_ch[2], num_classes=config['num_seg_classes'])
        
        # YOLO Head (Korrigierte Listen-Übergabe)
        self.yolo_head = YOLOHead(
            ch_dims=bb_ch,  # Ganze Liste
            num_classes=config['num_det_classes']
        )

    def forward(self, left, right=None):
        
        # 1. Backbone Features
        # Da Input schon [B, 3, H, W] ist, brauchen wir kein .repeat() mehr!
        # (Es schadet nicht, falls mal 1-Channel kommt, aber hier ist es unnötig)
        x_left = left.repeat(1, 3, 1, 1) if left.shape[1] == 1 else left
        fl = self.backbone(x_left) 

        # 2. Stereo Branch
        final_disp = None
        if right is not None:
            x_right = right.repeat(1, 3, 1, 1) if right.shape[1] == 1 else right
            fr = self.backbone(x_right)
            
            # --- FIX: RGB zu Grayscale für Refinement Guidance ---
            # Wir nehmen den Mean über die Channel-Dimension (dim=1)
            # Input: [B, 3, H, W] -> Output: [B, 1, H, W]
            l_img_gray = left.mean(dim=1, keepdim=True)
            
            # Jetzt füttern wir das 1-Kanal Bild in den Head
            final_disp = self.stereo_head(
                l_s8=fl[1], 
                r_s8=fr[1], 
                l_s4=fl[0], 
                l_img_raw=l_img_gray # <--- WICHTIG: Hier das graue Bild nutzen!
            )

        # 3. Segmentation Head
        seg_preds = self.seg_head(fl[0], fl[2])
        
        # 4. YOLO Head
        det_preds = self.yolo_head(fl[1], fl[2], fl[3])
        
        return final_disp, seg_preds, det_preds
# --- Initialization Utilities ---
def initialize_yolo_head(model, prior_prob=0.01):
    bias_value = -math.log((1 - prior_prob) / prior_prob)
    count = 0
    for m in model.modules():
        if isinstance(m, DecoupledHead):
            for layer in m.reg_conv:
                if isinstance(layer, ConvBR):
                    nn.init.kaiming_normal_(layer.conv.weight, mode='fan_out', nonlinearity='relu')
            nn.init.normal_(m.reg_pred.weight, mean=0.0, std=0.01)
            nn.init.constant_(m.reg_pred.bias, 0.0)
            
            for layer in m.cls_conv:
                if isinstance(layer, ConvBR):
                    nn.init.kaiming_normal_(layer.conv.weight, mode='fan_out', nonlinearity='relu')
            nn.init.normal_(m.cls_pred.weight, mean=0.0, std=0.01)
            nn.init.constant_(m.cls_pred.bias, bias_value)
            
            nn.init.normal_(m.obj_pred.weight, mean=0.0, std=0.01)
            nn.init.constant_(m.obj_pred.bias, bias_value)
            count += 1
    print(f"✅ Initialized {count} YOLO Heads with Prior Bias.")



DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FusedHexapodModel(CONFIG).to(DEVICE)
initialize_yolo_head(model)

# BatchNorm Momentum Fix
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.momentum = 0.01

print("✅ Fused StereoNet Hexapod Model ready.")

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


✅ Initialized 3 YOLO Heads with Prior Bias.
✅ Fused StereoNet Hexapod Model ready.


Cell 3: Unified Loss Function
This cell contains the corrected and unified loss function. It properly calculates the stereo loss on the refined output and combines it with segmentation and detection losses.

In [ ]:
# --- 1. Define Helper Blocks ---
class SimpleYOLOLoss(nn.Module):
    def __init__(self, num_classes, stride):
        super().__init__()
        self.num_classes = num_classes
        self.stride = stride
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        self.l1 = nn.L1Loss(reduction='none')

    def _get_targets(self, targets, cls_preds, reg_preds):
        """
        Erzeugt die Target-Tensoren für Objectness, Regression und Klassifizierung.
        
        Encoding:
        - Offsets (dx, dy): Subpixel-Lage innerhalb der Zelle [0, 1)
        - Größen (lw, lh): Logarithmierte Breite/Höhe in Grid-Einheiten
        """
        B, _, H, W = cls_preds.shape
        device = cls_preds.device
        
        # Initialisierung der Target-Tensoren auf dem korrekten Device
        cls_t = torch.zeros_like(cls_preds)
        reg_t = torch.zeros_like(reg_preds)
        obj_mask = torch.zeros((B, 1, H, W), device=device) 

        for b in range(B):
            # Prüfen, ob Boxen für diesen Batch-Eintrag existieren
            if targets[b].numel() == 0: 
                continue
            
            # targets[b] Format: [N, 5] -> [class, xc, yc, w, h] (normalisiert 0-1)
            gt_boxes = targets[b].clone()
            
            # Koordinaten von Normalisiert [0, 1] auf Grid-Skala [0, W/H] umrechnen
            gt_boxes[:, 1] *= W  # xc in Grid-Einheiten
            gt_boxes[:, 2] *= H  # yc in Grid-Einheiten
            gt_boxes[:, 3] *= W  # w  in Grid-Einheiten
            gt_boxes[:, 4] *= H  # h  in Grid-Einheiten
            
            for box in gt_boxes:
                cls_id, gx, gy, gw, gh = box.tolist()
                
                # Grid-Zelle bestimmen (Integer-Anteil)
                ix, iy = int(gx), int(gy)
                
                # Boundary Check: Liegt das Zentrum innerhalb des Grids?
                if 0 <= ix < W and 0 <= iy < H:
                    # 1. Objectness Target setzen (Zentrum der Box)
                    obj_mask[b, 0, iy, ix] = 1.0
                    
                    # 2. Regression Targets berechnen
                    # dx, dy: Abstand vom linken/oberen Rand der Zelle
                    dx = gx - ix
                    dy = gy - iy
                    
                    # lw, lh: Logarithmus der Grid-Größe für exp() Decoding im Head
                    # Epsilon verhindert log(0) bei extrem kleinen Boxen
                    lw = math.log(max(gw, 1e-6))
                    lh = math.log(max(gh, 1e-6))
                    
                    reg_t[b, 0, iy, ix] = dx
                    reg_t[b, 1, iy, ix] = dy
                    reg_t[b, 2, iy, ix] = lw
                    reg_t[b, 3, iy, ix] = lh
                    
                    # 3. Classification Target (One-Hot)
                    cid = int(cls_id)
                    if 0 <= cid < self.num_classes:
                        cls_t[b, cid, iy, ix] = 1.0

        return cls_t, reg_t, obj_mask

    def sigmoid_focal_loss(self, inputs, targets, alpha=0.25, gamma=2.0, reduction='none'):
        """Standard Focal Loss for dense detection."""
        p = torch.sigmoid(inputs)
        ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
        p_t = p * targets + (1 - p) * (1 - targets)
        loss = ce_loss * ((1 - p_t) ** gamma)

        if alpha >= 0:
            alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
            loss = alpha_t * loss

        if reduction == "mean":
            return loss.mean()
        elif reduction == "sum":
            return loss.sum()
        else:
            return loss

    def forward(self, preds, targets):
        # Slice components: [B, 5 + Num_Classes, H, W]
        reg_p = preds[:, :4, :, :]   # [dx, dy, log_w, log_h]
        obj_p = preds[:, 4:5, :, :]  # [objectness]
        cls_p = preds[:, 5:, :, :]   # [classes]

        # 1. Get Ground Truth Targets
        cls_t, reg_t, obj_mask = self._get_targets(targets, cls_p, reg_p)
        
        num_pos = torch.clamp(obj_mask.sum(), min=1.0)
        
        # 2. OBJECTNESS LOSS (mit Focal Loss für bessere Balance)
        l_obj = self.sigmoid_focal_loss(obj_p, obj_mask, reduction='sum') / num_pos

        # 3. REGRESSION LOSS (nur auf positiven Samples)
        # WICHTIG: Separate Gewichtung für Offsets vs. Größen
        l_offset = (self.l1(reg_p[:, :2], reg_t[:, :2]) * obj_mask).sum() / num_pos
        l_size = (self.l1(reg_p[:, 2:], reg_t[:, 2:]) * obj_mask).sum() / num_pos
        l_box = l_offset + 2.0 * l_size  # Größen sind wichtiger

        # 4. CLASSIFICATION LOSS
        l_cls = self.sigmoid_focal_loss(cls_p, cls_t, reduction='sum') / num_pos

        # ANGEPASSTE Gewichtung: Box-Loss ist jetzt wichtiger
        return {
            'total': 5.0 * l_box + 20.0 * l_obj + l_cls,
            'box': l_box.item(),
            'obj': l_obj.item(),
            'cls': l_cls.item()
        }


class EdgeAwareSmoothnessLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, disp, img):
        mean_disp = disp.mean(2, True).mean(3, True)
        disp = disp / (mean_disp + 1e-7)
        
        grad_disp_x = torch.abs(disp[:, :, :, :-1] - disp[:, :, :, 1:])
        grad_disp_y = torch.abs(disp[:, :, :-1, :] - disp[:, :, 1:, :])

        grad_img_x = torch.mean(torch.abs(img[:, :, :, :-1] - img[:, :, :, 1:]), 1, keepdim=True)
        grad_img_y = torch.mean(torch.abs(img[:, :, :-1, :] - img[:, :, 1:, :]), 1, keepdim=True)

        grad_img_x = torch.exp(-torch.mean(grad_img_x, 1, keepdim=True))
        grad_img_y = torch.exp(-torch.mean(grad_img_y, 1, keepdim=True))

        return torch.mean(grad_disp_x * grad_img_x) + torch.mean(grad_disp_y * grad_img_y)


# --- 2. Main Loss Class ---
class FusedHexapodLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        # YOLO Skalen-Verluste
        self.yolo_s8_loss  = SimpleYOLOLoss(config['num_det_classes'], stride=8)
        self.yolo_s16_loss = SimpleYOLOLoss(config['num_det_classes'], stride=16)
        self.yolo_s32_loss = SimpleYOLOLoss(config['num_det_classes'], stride=32)
        
        # KEIN seg_loss (CrossEntropy gegen GT) mehr hier!
        self.kl_div = nn.KLDivLoss(reduction='batchmean') 
        self.smooth_loss = EdgeAwareSmoothnessLoss()
        
        # Basis-Gewichte (werden durch UncertaintyWeighting modifiziert)
        self.w_yolo = 1.0 
        self.w_kd = 1.0     # Fokus liegt jetzt auf KD für Segmentation
        self.w_stereo = 1.0

    def forward(self, preds, targets, teacher_preds, left_img=None):
        # Entpacken der Vorhersagen
        stereo_preds, seg_preds, det_preds = preds
        
        task_tensors = {} # Für Gradienten (Tensors)
        logs = {}         # Für Tensorboard (Floats)

        # --- 1. STEREO LOSS ---
        if stereo_preds is not None and targets.get('disp') is not None:
            gt_disp = targets['disp']
            if gt_disp.dim() == 3: gt_disp = gt_disp.unsqueeze(1)
            
            mask = (gt_disp > 0) & (gt_disp < CONFIG['max_disp_pixel'])
            if mask.sum() > 0:
                l_stereo = F.smooth_l1_loss(stereo_preds[mask], gt_disp[mask])
                
                # Edge-Aware Smoothness hinzufügen
                if left_img is not None:
                    l_smooth = self.smooth_loss(stereo_preds, left_img)
                    l_stereo = l_stereo + 0.1 * l_smooth
                    logs['smooth'] = l_smooth.item()
                
                task_tensors['stereo'] = l_stereo
                logs['stereo'] = l_stereo.item()

        # --- 2. KNOWLEDGE DISTILLATION (KD) ---
        if teacher_preds is not None and seg_preds is not None:
            T = 2.0 
            
            s_logits = F.interpolate(seg_preds, size=teacher_preds.shape[-2:], 
                                    mode='bilinear', align_corners=False)
            
            # STUDENT: Muss Log-Softmax sein (Korrekt)
            p_s = F.log_softmax(s_logits / T, dim=1)
            
            # TEACHER: Da der Teacher bereits LOG-Werte liefert, 
            # müssen wir erst exp() nehmen, um auf Probs zu kommen, 
            # und DANN den Softmax für die Temperatur-Skalierung anwenden.
            # Alternativ: Direkt die Log-Werte skalieren.
            
            # Sicherster Weg:
            p_t_base = torch.exp(teacher_preds) # Zurück zu [0, 1]
            p_t = F.softmax(torch.log(p_t_base + 1e-7) / T, dim=1)
            
            # KL Divergenz
            # WICHTIG: PyTorch KLDivLoss braucht (input_log_probs, target_probs)
            l_kd = self.kl_div(p_s, p_t) * (T**2)
            
            task_tensors['seg'] = l_kd 
            logs['seg_kd'] = l_kd.item()

        # --- 3. YOLO LOSS ---
        if det_preds is not None and targets.get('det') is not None:
            gt_det = targets['det']
            
            # Alle 3 Skalen berechnen
            r8  = self.yolo_s8_loss(det_preds[0], gt_det)
            r16 = self.yolo_s16_loss(det_preds[1], gt_det)
            r32 = self.yolo_s32_loss(det_preds[2], gt_det)
            
            l_yolo = (r8['total'] + r16['total'] + r32['total']) / 3.0
            
            task_tensors['yolo'] = l_yolo
            logs['yolo'] = l_yolo.item()
            logs['yolo_box'] = (r8['box'] + r16['box'] + r32['box']) / 3.0
            logs['yolo_obj'] = (r8['obj'] + r16['obj'] + r32['obj']) / 3.0

        return task_tensors, logs
print("✅ Loss function updated.")

✅ Loss function updated.


Cell 4: Data Parsing Helpers
This cell contains the necessary helper functions to scan the respective dataset directories (FlyingThings3D, TartanAir, and COCO) and create a unified list of file paths and labels. The RealFusedDataset class in the next cell will use these functions.

In [14]:
import os
import glob
import numpy as np
from pycocotools.coco import COCO

def parse_ft3d(root):
    """ 
    Scans the FlyingThings3D directory for stereo pairs and disparity maps. 
    Adapted for flattened structure: .../image_clean/left/*.png
    """
    print("   Scanning FlyingThings3D...")
    samples = []
    
    # Pfade basierend auf deiner Struktur
    img_dir_l = os.path.join(root, 'frames_cleanpass', 'TRAIN', 'image_clean', 'left')
    disp_dir_l = os.path.join(root, 'disparity', 'TRAIN', 'disparity', 'left')
    
    if not os.path.exists(img_dir_l):
        print(f"   [Error] Could not find directory: {img_dir_l}")
        return []
    
    # Glob alle PNGs
    left_files = sorted(glob.glob(os.path.join(img_dir_l, '*.png')))
    
    if not left_files:
        print(f"   [Warning] Directory found but no .png files inside: {img_dir_l}")
        return []

    # Matching mit Rechts & Disparity
    for l_path in left_files:
        filename = os.path.basename(l_path)
        
        # Rechtes Bild finden (String Replace 'left' -> 'right')
        # Vorsicht: Wir ersetzen nur das letzte Vorkommen oder nutzen os.sep
        parent_dir = os.path.dirname(l_path)
        if 'left' in parent_dir:
            r_path = l_path.replace(os.sep + 'left' + os.sep, os.sep + 'right' + os.sep)
        else:
            # Fallback, falls Pfadstruktur anders ist
            r_path = l_path.replace('left', 'right')
        
        # Disparity Pfad (.pfm)
        d_filename = filename.replace('.png', '.pfm')
        d_path = os.path.join(disp_dir_l, d_filename)
        
        if os.path.exists(r_path) and os.path.exists(d_path):
            samples.append({
                'type': 'stereo',
                'source': 'ft3d',
                'l': l_path,
                'r': r_path,
                'd': d_path,
                's': None, # Kein Seg
                'b': None  # Keine Boxen
            })
            
    print(f"   -> Found {len(samples)} FT3D pairs.")
    return samples

def parse_tartan(root):
    """ Scans the TartanAir directory for stereo pairs, depth, and optional segmentation. """
    print("   Scanning TartanAir...")
    samples = []
    
    # Suche rekursiv nach 'image_left' Ordnern
    # TartanAir Struktur: .../Environment/Easy/P000/image_left/...
    search_pattern = os.path.join(root, '**', 'image_left')
    left_folders = glob.glob(search_pattern, recursive=True)
    
    if not left_folders:
        print(f"   [Warning] No 'image_left' folders found in {root}")
        return []

    for l_folder in left_folders:
        # Parent ist z.B. .../P000/
        parent = os.path.dirname(l_folder)
        
        # Iteriere über Dateien im linken Ordner
        for f in os.listdir(l_folder):
            if not f.endswith('.png'): continue
            
            l_path = os.path.join(l_folder, f)
            
            # TartanAir Naming Convention: 
            # Left:  000000_left.png
            # Right: 000000_right.png
            r_filename = f.replace('_left', '_right')
            r_path = os.path.join(parent, 'image_right', r_filename)
            
            # Depth: 000000_left_depth.npy
            d_filename = f.replace('.png', '_depth.npy')
            d_path = os.path.join(parent, 'depth_left', d_filename)
            
            # Seg: 000000_left_seg.npy
            s_filename = f.replace('.png', '_seg.npy')
            s_path = os.path.join(parent, 'seg_left', s_filename)
            
            if os.path.exists(r_path) and os.path.exists(d_path):
                samples.append({
                    'type': 'stereo', 
                    'source': 'tartan',
                    'l': l_path, 
                    'r': r_path, 
                    'd': d_path, 
                    's': s_path if os.path.exists(s_path) else None, 
                    'b': None
                })
                
    print(f"   -> Found {len(samples)} TartanAir samples.")
    return samples

def parse_coco(root):
    """ Scans the COCO directory for images and bounding box annotations. """
    print("   Scanning COCO 2017...")
    samples = []
    
    # Pfade prüfen
    ann_file = os.path.join(root, 'annotations', 'instances_train2017.json')
    img_dir = os.path.join(root, 'train2017')
    
    if not os.path.exists(ann_file): 
        print(f"   [Error] Annotation file not found: {ann_file}")
        return []
    
    try:
        coco = COCO(ann_file)
    except Exception as e:
        print(f"   [Error] Failed to load COCO JSON: {e}")
        return []

    # Kategorien filtern? (Optional, hier nehmen wir alle)
    # cat_ids = coco.getCatIds(catNms=['person', 'car', ...])
    # img_ids = coco.getImgIds(catIds=cat_ids)
    img_ids = coco.getImgIds()

    for iid in img_ids:
        img_info = coco.loadImgs(iid)[0]
        path = os.path.join(img_dir, img_info['file_name'])
        
        if not os.path.exists(path): continue
        
        # Annotations laden
        ann_ids = coco.getAnnIds(imgIds=iid, iscrowd=False)
        anns = coco.loadAnns(ann_ids)
        
        boxes = []
        H, W = img_info['height'], img_info['width']
        
        for ann in anns:
            # COCO bbox: [x_top_left, y_top_left, width, height]
            x, y, w, h = ann['bbox']
            
            # Validierung: Keine leeren Boxen
            if w < 1 or h < 1: continue

            # Convert to YOLO format [class, x_center, y_center, width, height] (Normalized)
            # category_id in COCO ist nicht kontinuierlich (1...90), wir müssen mappen falls nötig.
            # Hier: Wir nehmen an, num_classes deckt max(category_id) ab oder wir mappen später.
            # Einfaches Mapping: id - 1 (Vorsicht bei COCO, manche IDs fehlen!)
            cls_id = ann['category_id'] - 1 
            
            # YOLO Format: Center-based
            cx = (x + w / 2.0) / W
            cy = (y + h / 2.0) / H
            nw = w / W
            nh = h / H
            
            boxes.append([cls_id, cx, cy, nw, nh])
            
        if boxes:
            # Mono-Sample (nur Links, kein Rechts/Disp)
            samples.append({
                'type': 'mono',
                'source': 'coco',
                'l': path,
                'b': np.array(boxes, dtype=np.float32),
                'r': None,
                'd': None,
                's': None
            })
            
    print(f"   -> Found {len(samples)} COCO samples.")
    return samples

print("✅ Data parsing helper functions are ready.")


✅ Data parsing helper functions are ready.


Cell 5: Data Loading & Augmentation
This cell defines the RealFusedDataset class. It uses the parsers from the previous cell to build a master file list and then applies appropriate augmentations for training. It also includes the corrected .pfm file reader.

In [ ]:
def read_pfm_fixed(file_path):
    """ Reads a .pfm file and returns a numpy array. Includes scaling. """
    with open(file_path, 'rb') as f:
        header = f.readline().decode().rstrip()
        color = (header == 'PF')
        dim_match = re.match(r'^(\d+)\s(\d+)\s$', f.readline().decode('utf-8'))
        width, height = map(int, dim_match.groups())
        scale = float(f.readline().decode().rstrip())
        endian = '<' if scale < 0 else '>'
        scale = abs(scale)
        
        data = np.fromfile(f, endian + 'f')
        shape = (height, width, 3) if color else (height, width)
        data = np.reshape(data, shape)
        data = np.flipud(data) 
    return (data * scale).copy()

class RealFusedDataset(Dataset):
    def __init__(self, roots, mode='train', img_size=(480, 640), use_teacher=True, teacher_instance=None):
        self.img_h, self.img_w = img_size
        self.use_teacher = use_teacher
        self.teacher_instance = teacher_instance
        self.mode = mode
        self.samples = []
        
        # 1. Daten laden (FT3D ist raus!)
        if 'tartan' in roots: self.samples.extend(parse_tartan(roots['tartan']))
        if 'coco' in roots: self.samples.extend(parse_coco(roots['coco']))
        
        # Sampler-Vorbereitung: Wir trennen COCO und Rest
        self.coco_indices = [i for i, s in enumerate(self.samples) if s['source'] == 'coco']
        self.tartan_indices = [i for i, s in enumerate(self.samples) if s['source'] == 'tartan']
        
        print(f"📊 Dataset ({mode}): {len(self.coco_indices)} COCO | {len(self.tartan_indices)} TartanAir")
        
        # 1. Main Augmentation (Left Image + Boxes)
        self.transform_geom = A.Compose([
            A.Resize(height=self.img_h, width=self.img_w),
            # Optional: A.HorizontalFlip(p=0.5) falls gewünscht
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.1))

        # 2. Right Image Augmentation (No Boxes)
        self.transform_geom_no_boxes = A.Compose([
            A.Resize(height=self.img_h, width=self.img_w),
        ])

        # ImageNet Stats for normalization
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.samples)

    def get_pseudo_labels_for_sample(self, img_np):
        """ Hilfsfunktion: Nutzt den Teacher, um Boxen für ein Numpy-Bild zu generieren """
        if self.teacher_instance is None:
            return torch.zeros((0, 5))
        
        # Das Bild img_np (H, W, C) wird vom Teacher verarbeitet
        pseudo_boxes = self.teacher_instance.get_pseudo_labels(img_np)
        
        if pseudo_boxes is not None and len(pseudo_boxes) > 0:
            return torch.tensor(pseudo_boxes, dtype=torch.float32)
        return torch.zeros((0, 5))

    def __getitem__(self, idx):
        # 1. BALANCED SAMPLING (Wichtig für YOLO Stabilität)
        # Im Training erzwingen wir in 50% der Fälle ein echtes COCO-Bild
        if self.mode == 'train' and len(self.coco_indices) > 0:
            if torch.rand(1).item() > 0.5:
                idx = np.random.choice(self.coco_indices)
        
        s = self.samples[idx]
        
        # 2. BILDER LADEN (BGR -> RGB)
        img_l_raw = cv2.imread(s['l'])
        img_l_rgb = cv2.cvtColor(img_l_raw, cv2.COLOR_BGR2RGB)
        
        if s.get('r'):
            img_r_rgb = cv2.cvtColor(cv2.imread(s['r']), cv2.COLOR_BGR2RGB)
        else:
            img_r_rgb = img_l_rgb.copy() # Mono-Fallback

        h, w, _ = img_l_rgb.shape

        # 3. GEOMETRIE LADEN (Disparity / Depth)
        if s.get('d') and s['d'].endswith('.pfm'):
             raw_geo = read_pfm_fixed(s['d']) 
        elif s.get('d'):
             raw_geo = np.load(s['d'])
        else:
             raw_geo = np.full((h, w), -1.0, dtype=np.float32)

        # TartanAir Fix: Depth -> Disparity
        if s.get('source') == 'tartan':
            valid_mask = raw_geo > 1e-4 
            disp = np.zeros_like(raw_geo)
            disp[valid_mask] = 80.0 / raw_geo[valid_mask] 
            disp = np.clip(disp, 0, 192)
        else:
            disp = raw_geo

        # Segmentation Maske laden (falls vorhanden)
        seg = np.load(s['s']) if s.get('s') else np.full((h, w), 255, dtype=np.uint8)
        
        # 4. GROUND TRUTH BOXEN (Nur für COCO relevant)
        boxes, labels = [], []
        if s['source'] == 'coco' and s.get('b') is not None:
            raw_boxes = np.array(s['b'])
            if len(raw_boxes) > 0:
                valid_b = (raw_boxes[:, 3] > 1e-4) & (raw_boxes[:, 4] > 1e-4)
                clean_boxes = raw_boxes[valid_b]
                if len(clean_boxes) > 0:
                    clean_boxes[:, 1:] = np.clip(clean_boxes[:, 1:], 0.0, 1.0)
                    labels = clean_boxes[:, 0].tolist()
                    boxes = clean_boxes[:, 1:].tolist()

        # 5. AUGMENTATION ANWENDEN
        # Wir augmentieren erst das Bild, damit der Teacher das gleiche Bild wie der Student sieht
        aug_l = self.transform_geom(image=img_l_rgb, mask=disp, masks=[seg], bboxes=boxes, class_labels=labels)
        
        l_img_aug = aug_l['image']      # Numpy HWC
        disp_aug = aug_l['mask']        # Numpy HW
        seg_aug = aug_l['masks'][0]     # Numpy HW
        boxes_aug = aug_l['bboxes']     # List (nur COCO)
        labels_aug = aug_l['class_labels']

        # Rechtes Bild synchron (Resize)
        aug_r = self.transform_geom_no_boxes(image=img_r_rgb)['image']
        
        # 6. KONVERTIERUNG ZU TENSOR & ROBOT-INPUT
        def to_tensor(img):
            return torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        t_l_rgb = to_tensor(l_img_aug) 
        t_r_rgb = to_tensor(aug_r)     
        
        # Teacher Input (RGB)
        teacher_input = (t_l_rgb - self.mean) / self.std

        # Robot Input (Grayscale -> 3ch)
        gray_l = 0.299 * t_l_rgb[0] + 0.587 * t_l_rgb[1] + 0.114 * t_l_rgb[2]
        gray_r = 0.299 * t_r_rgb[0] + 0.587 * t_r_rgb[1] + 0.114 * t_r_rgb[2]
        
        robot_l = (gray_l.unsqueeze(0).repeat(3, 1, 1) - self.mean) / self.std
        robot_r = (gray_r.unsqueeze(0).repeat(3, 1, 1) - self.mean) / self.std

        # 7. YOLO TARGET SELECTION (GT vs. Pseudo-Label)
        if s['source'] == 'coco':
            # Echte Labels nutzen
            if boxes_aug:
                t_det = torch.tensor([[l] + list(b) for l, b in zip(labels_aug, boxes_aug)], dtype=torch.float32)
            else:
                t_det = torch.zeros((0, 5))
        elif s['source'] == 'tartan' and self.use_teacher:
            # Pseudo-Labels vom Teacher für TartanAir generieren
            # Wir übergeben das bereits augmentierte Bild!
            t_det = self.get_pseudo_labels_for_sample(l_img_aug) 
        else:
            t_det = torch.zeros((0, 5))

        # 8. OUTPUT DICTIONARY
        return {
            'left': robot_l, 
            'right': robot_r, 
            'teacher': teacher_input,
            'disp': torch.from_numpy(disp_aug).unsqueeze(0).float(), 
            'seg': torch.from_numpy(seg_aug).long(), 
            'det': t_det,
            'is_pseudo': (s['source'] == 'tartan')
        }


Cell6: The Hexapod Visualizer
This function takes a batch and the model predictions, then displays the Left Image, Stereo Disparity, Semantic Map (Hexapod Physics), and YOLO Detections side-by-side.

In [ ]:
import matplotlib.colors as mcolors

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import torch

def visualize_hexapod_output(step, writer):
    """
    Dein Original-Dashboard.
    Layout: 2 Zeilen, 4 Spalten.
    Zusatz: Deep-Debug Prints im Terminal, um Model-Output zu validieren.
    """
    was_training = model.training
    model.eval()
    
    # Threshold global definiert für Konsistenz zwischen Print und Bild
    VIS_THRESH = 0.15 
    
    with torch.no_grad():
        fig, axes = plt.subplots(2, 4, figsize=(24, 12))
        
        for row, (name, b) in enumerate(static_batches.items()):
            l = b['left'].to(DEVICE)
            r = b['right'].to(DEVICE)
            t_img = b['teacher'].to(DEVICE)
            
            # Forward pass
            final_disp, seg_preds, det_preds = model(l, r)
            
            # --- COLUMN 1: GT / INPUT ---
            ax1 = axes[row, 0]
            img_bg = l[0].permute(1,2,0).cpu().numpy()
            # Normalisierung für Anzeige
            img_bg = (img_bg - img_bg.min()) / (img_bg.max() - img_bg.min() + 1e-5)
            
            if name == 'yolo':
                ax1.imshow(img_bg)
                h, w = img_bg.shape[:2]
                if 'det' in b:
                    for box in b['det'][0]:
                        cls_id, xc, yc, bw, bh = box.tolist()
                        x1, y1 = (xc - bw/2)*w, (yc - bh/2)*h
                        box_color = plt.cm.tab10(int(cls_id) % 10)
                        rect = plt.Rectangle((x1, y1), bw*w, bh*h, fill=False, color=box_color, linewidth=2)
                        ax1.add_patch(rect)
                ax1.set_title("GT YOLO BOXES (Input)")
                vmax_scale = CONFIG['max_disp_pixel']
            else:
                gt_disp = b['disp'][0].cpu().numpy().squeeze()
                valid_mask = (gt_disp > 0) & (gt_disp < CONFIG['max_disp_pixel'])
                vmax_scale = np.percentile(gt_disp[valid_mask], 98) if valid_mask.any() else CONFIG['max_disp_pixel']
                im1 = ax1.imshow(gt_disp, cmap='magma', vmin=0, vmax=vmax_scale)
                ax1.set_title(f"GT DISPARITY (Scale: {vmax_scale:.1f})")
                plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

            # --- COLUMN 2: CORE PREDICTION ---
            ax2 = axes[row, 1]
            if name == 'yolo':
                yolo_map = det_preds[0][0] # Scale 8
                obj_score = torch.sigmoid(yolo_map[4, :, :])
                cls_score = torch.sigmoid(yolo_map[5:, :, :]).max(dim=0)[0]
                conf_map = (obj_score * cls_score).cpu().numpy()
                im2 = ax2.imshow(conf_map, cmap='viridis', vmin=0, vmax=1)
                ax2.set_title(f"YOLO CONF (Max: {conf_map.max():.2f})")
                plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
            else:
                pred_disp = final_disp[0, 0].cpu().numpy()
                im2 = ax2.imshow(pred_disp, cmap='magma', vmin=0, vmax=vmax_scale)
                ax2.set_title(f"PRED DISPARITY (Soft-Argmax)")
                plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

            # --- COLUMN 3: STUDENT SEGMENTATION ---
            ax3 = axes[row, 2]
            student_seg = torch.argmax(seg_preds, dim=1)[0].cpu().numpy()
            im3 = ax3.imshow(student_seg, cmap='tab10', vmin=0, vmax=9)
            ax3.set_title("STUDENT PHYSICS SEG")
            if row == 0:
                plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04, ticks=range(10))

            # --- COLUMN 4: DEBUG / TEACHER ---
            ax4 = axes[row, 3]
            if name == 'yolo':
                ax4.imshow(img_bg)
                h, w = img_bg.shape[:2]
                yolo_map = det_preds[0][0] # Scale 8
                stride = 8 
                
                # Dekodierung
                obj_score = torch.sigmoid(yolo_map[4, :, :])
                cls_probs = torch.sigmoid(yolo_map[5:, :, :])
                final_conf = obj_score * torch.max(cls_probs, dim=0)[0]
                
                mask = final_conf > VIS_THRESH
                num_candidates = mask.sum().item()
                boxes_drawn = 0
               
                if num_candidates > 0:
                    ys, xs = torch.where(mask)
                    for i in range(len(xs)):
                        gx, gy = xs[i].item(), ys[i].item()
                        
                        dx = torch.sigmoid(yolo_map[0, gy, gx]).item()
                        dy = torch.sigmoid(yolo_map[1, gy, gx]).item()
                        gw = torch.exp(torch.clamp(yolo_map[2, gy, gx], -5, 5)).item()
                        gh = torch.exp(torch.clamp(yolo_map[3, gy, gx], -5, 5)).item()
                        
                        cx_pixel = (gx + dx) * stride
                        cy_pixel = (gy + dy) * stride
                        bw_pixel = gw * stride
                        bh_pixel = gh * stride
                        
                        x1 = cx_pixel - bw_pixel / 2
                        y1 = cy_pixel - bh_pixel / 2
                        
                        cls_id = torch.argmax(cls_probs[:, gy, gx]).item()
                        box_color = plt.cm.tab10(int(cls_id) % 10)
                        
                        # Alpha (Optional: macht es hübscher, kannst du rausnehmen wenn nicht gewünscht)
                        alpha = min(1.0, final_conf[gy, gx].item() * 5.0) 

                        rect = plt.Rectangle((x1, y1), bw_pixel, bh_pixel, fill=False, edgecolor=box_color, linewidth=2, alpha=alpha)
                        ax4.add_patch(rect)
                        boxes_drawn += 1
                
                ax4.set_title(f"PRED BOXES (Sigmoid, n={boxes_drawn})")
            else:
                with torch.no_grad():
                    teacher_out = teacher_seg(t_img)
                    teacher_physics = torch.argmax(teacher_out, dim=1)[0].cpu().numpy()
                ax4.imshow(teacher_physics, cmap='tab10', vmin=0, vmax=9)
                ax4.set_title("TEACHER TARGET")

        for ax in axes.flatten():
            ax.set_xticks([]); ax.set_yticks([])
        
        plt.tight_layout()
        writer.add_figure('Model_Progress/Deep_Vision_Dashboard', fig, global_step=step)
        writer.add_scalar('Model_Progress/YOLO_Final_CONF', final_conf.max().item(), global_step=step)
        writer.add_scalar('Model_Progress/YOLO_Num_Candidates', num_candidates, global_step=step)
        writer.add_scalar('Model_Progress/YOLO_Boxes_Drawn', boxes_drawn, global_step=step)
        plt.close(fig)
        if was_training: model.train()

print("✅ Visualization updated: YOLO uses Sigmoid-Decoding and Stereo uses Soft-Argmax scale.")

✅ Visualization updated: YOLO uses Sigmoid-Decoding and Stereo uses Soft-Argmax scale.


Cell 7: Model Graph
1. The Shared Backbone: Where the image features are first extracted.
2. The Split: Where the data branches off into three different "heads" (Stereo, Segmentation, and YOLO).
3. The Stereo Neck: How the left and right features are concatenated or subtracted to form a cost volume.
4. Tensor Shapes: Most importantly, it labels the dimensions (e.g., $1 \times 120 \times 160$) at every step, which helps you verify your downsampling logic is working correctly.

In [17]:
# Create a dummy input matching your robot's input size
dummy_left = torch.randn(1, 1, CONFIG['img_height'], CONFIG['img_width']).to(DEVICE)
dummy_right = torch.randn(1, 1, CONFIG['img_height'], CONFIG['img_width']).to(DEVICE)

# Add the graph to TensorBoard
writer = SummaryWriter("./logs")

class GraphWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, l, r):
        # stereo is now the SINGLE final disparity tensor
        stereo, seg, det = self.model(l, r)
        
        # We just return the main outputs we care about visualizing
        # stereo: [B, 1, H, W]
        # seg: [B, 10, H/4, W/4]
        # det: Tuple of lists... let's just grab the first scale for the graph
        
        # Return: Final Disparity, Segmentation, and YOLO Scale 0 (Class+Box)
        return stereo, seg, det[0][0]

# 1. Create the wrapper
wrapper = GraphWrapper(model).to(DEVICE)

# 2. Use the wrapper for the graph
print("📊 Generating model graph for TensorBoard (using wrapper)...")
writer.add_graph(wrapper, [dummy_left, dummy_right])
print("✅ Model graph for TensorBoard done...")
# 3. Clean up memory
del wrapper
torch.cuda.empty_cache()

📊 Generating model graph for TensorBoard (using wrapper)...


/tmp/ipykernel_64954/2539891618.py:264: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  x_left = left.repeat(1, 3, 1, 1) if left.shape[1] == 1 else left
/tmp/ipykernel_64954/2539891618.py:270: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  x_right = right.repeat(1, 3, 1, 1) if right.shape[1] == 1 else right
/tmp/ipykernel_64954/2539891618.py:128: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means th

✅ Model graph for TensorBoard done...


Cell 8: Loss-Wrapper with trainable weights
This class encapsulates FusedHexapodLoss and mangages the weights as trainable parameters.

In [ ]:
import torch.nn as nn

class UncertaintyWeighting(nn.Module):
    def __init__(self, base_criterion):
        super().__init__()
        self.base_criterion = base_criterion
        
        # stereo (reg), seg (cls via KD), yolo (cls)
        self.log_vars = nn.Parameter(torch.zeros(3))

    def forward(self, preds, targets, teacher_preds, left_img=None, step=0):
        task_tensors, logs = self.base_criterion(
            preds, targets, teacher_preds, left_img
        )

        weighted_loss = 0.0
        weighted_logs = {}
        keys = ['stereo', 'seg', 'yolo']

        # --- WARMUP ---
        if step < 2000:
            w = {'stereo': 1.0, 'seg': 1.0, 'yolo': 1.0}
            for key in keys:
                if key in task_tensors:
                    weighted_loss += w[key] * task_tensors[key]
                    weighted_logs[f"weight_{key}"] = w[key]

        else:
            # --- KENDALL ---
            for i, key in enumerate(keys):
                if key not in task_tensors:
                    continue

                v = self.log_vars[i].clamp(-1.0, 3.0)
                precision = torch.exp(-v)
                loss = task_tensors[key]

                if key == 'stereo':
                    # Regression
                    weighted_loss += 0.5 * precision * loss + 0.5 * v
                else:
                    # Klassifikation (Seg via KD, YOLO)
                    weighted_loss += precision * loss + v

                weighted_logs[f"weight_{key}"] = precision.item()

        weighted_logs.update(logs)
        return weighted_loss, weighted_logs

def get_balanced_optimizer(model, criterion, base_lr=1e-4):
    # Parameter Gruppen
    backbone_params = model.backbone.parameters()
    yolo_params = model.yolo_head.parameters()
    stereo_params = model.stereo_head.parameters()
    seg_params = model.seg_head.parameters()

    params = [
        # Backbone: Der "Dienstleister" für alle (sehr langsam lernen)
        {'params': backbone_params, 'lr': base_lr * 0.1, 'weight_decay': 1e-5},
        
        # YOLO: Darf schnell konvergieren, aber nicht dominieren
        {'params': yolo_params, 'lr': base_lr * 0.5}, # Reduziert, da Teacher stabilisiert
        
        # Stereo & Seg: Brauchen mehr Zeit für Details
        {'params': stereo_params, 'lr': base_lr * 1.2},
        {'params': seg_params, 'lr': base_lr * 1.0},
        
        # UW Gewichte
        {'params': criterion.parameters(), 'lr': base_lr}
    ]
    return torch.optim.AdamW(params)

Cell 9: The Final Training Loop
This cell sets up and executes the main training process. It includes the crucial hexapod_collate function to handle mixed-data batches, initializes the data loaders, optimizer, and scheduler, and contains the complete training and validation logic with logging to TensorBoard.

In [ ]:
# --- Segmentation Teacher Model ---
class SegmentationTeacher(nn.Module):
#class UnifiedHexapodTeacher(nn.Module):
    """
    A 10-Class Teacher that bridges Indoor (ADE20K) and Outdoor (ADE20K/TartanAir)
    semantics for a Hexapod Robot.
    """
    def __init__(self, device='cuda'):
        super().__init__()
        # Using SegFormer B4 for high accuracy on texture details (Grass vs Carpet)
        model_name = "nvidia/segformer-b4-finetuned-ade-512-512"
        self.model = SegformerForSemanticSegmentation.from_pretrained(model_name)
        self.model.eval()
        self.model.to(device)
        
        for param in self.model.parameters():
            param.requires_grad = False

        self.num_source_classes = 150
        self.num_target_classes = 10  # Unified List
        
        map_matrix = torch.zeros(self.num_source_classes, self.num_target_classes)

        # --- 1. HARD_FLAT (Class 0) ---
        # Indoor: Floor(3), Wood(19)
        # Outdoor: Road(6), Sidewalk(11), Path(16), Platform(53), Flooring(105)
        # Physics: Rigid, predictable friction.
        hard_indices = [3, 19, 6, 11, 16, 53, 105]
        map_matrix[hard_indices, 0] = 1.0

        # --- 2. SOFT_FLAT (Class 1) ---
        # Indoor: Rug(29), Carpet(57)
        # Physics: High friction, but generally flat.
        soft_flat_indices = [29, 57]
        map_matrix[soft_flat_indices, 1] = 1.0

        # --- 3. NATURAL_UNEVEN (Class 2) ---
        # Outdoor: Grass(9), Earth(13), Field(28), Sand(46), Soil(94), Land(93), Rock(122 - gravel)
        # Physics: Compliant (sinking feet), uneven height. Needs High-Step.
        natural_indices = [9, 13, 28, 46, 94, 93, 122]
        map_matrix[natural_indices, 2] = 1.0

        # --- 4. WATER (Class 3) ---
        # Water(21), Sea(26), River(60), Lake(103), Pool(128)
        map_matrix[[21, 26, 60, 103, 128], 3] = 1.0

        # --- 5. CLUTTER (Class 4) ---
        # Step-over items: Pillow(59), Box(70), Paper(96), Towel(100), Clothes(106), Bag(112)
        # Note: In TartanAir, this corresponds to small scattering objects.
        clutter_indices = [59, 70, 96, 100, 106, 112]
        map_matrix[clutter_indices, 4] = 1.0

        # --- 6. STAIRS (Class 5) ---
        # Stairs(25), Step(126), Escalator(124)
        map_matrix[[25, 126, 124], 5] = 1.0

        # --- 7. OBSTACLES (Class 6) ---
        # Structural: Wall(0), Building(1), Fence(32), Railing(104)
        # Furniture: Cabinet(10), Bed(7), Chair(19), Sofa(23), Table(15), Shelf(41)
        # Outdoor: Tree(4), Plant(17 - bushes)
        obs_indices = [0, 1, 32, 104, 10, 7, 19, 23, 15, 41, 4, 17]
        map_matrix[obs_indices, 6] = 1.0

        # --- 8. GLASS (Class 7) ---
        # Window(8), Glass(63), Mirror(66)
        map_matrix[[8, 63, 66], 7] = 1.0

        # --- 9. DYNAMIC (Class 8) ---
        # Person(12), Car(20), Bus(80), Bicycle(127)
        map_matrix[[12, 20, 80, 127], 8] = 1.0

        # --- 10. SKY/VOID (Class 9) ---
        # Sky(2), Ceiling(5), Light(82)
        map_matrix[[2, 5, 82], 9] = 1.0

        # Fill remaining unassigned classes to OBSTACLES
        current_assigned = map_matrix.sum(dim=1)
        unassigned_indices = (current_assigned == 0).nonzero(as_tuple=True)[0]
        map_matrix[unassigned_indices, 6] = 1.0 
        
        self.register_buffer('map_matrix', map_matrix)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))

    def forward(self, x):
        with torch.no_grad():
            mean = self.mean.to(x.device)
            std = self.std.to(x.device)
            map_matrix = self.map_matrix.to(x.device)
            x_norm = (x - mean) / std
            outputs = self.model(x_norm)
            probs_150 = F.softmax(outputs.logits, dim=1).permute(0, 2, 3, 1)
            probs_10 = torch.matmul(probs_150, map_matrix).permute(0, 3, 1, 2)
            
            return torch.log(probs_10 + 1e-6)

import torch

class YoloTeacher:
    def __init__(self, device):
        # Wir nutzen ein robustes Modell (v5m oder v8m) als Wissensquelle
        self.model = torch.hub.load('ultralytics/yolov5', 'yolov5m', pretrained=True).to(device).eval()
        self.device = device

    @torch.no_grad()
    def get_pseudo_labels(self, img_np):
        """
        img_np: [H, W, 3] Numpy Array (0-255) von Albumentations
        """
        # Konvertierung zu Tensor und Normalisierung auf [0, 1] für den internen Check
        img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).float().to(self.device) / 255.0
        
        # YOLOv5 hub model erwartet oft [0, 1] oder [0, 255] je nach Version.
        # Sicherste Variante für torch.hub: 
        results = self.model(img_tensor.unsqueeze(0)) # Viele hub-Modelle skalieren intern selbst
        
        preds = results.xywhn[0].cpu().numpy() 
        valid_preds = preds[preds[:, 4] > 0.4]
        
        if len(valid_preds) == 0:
            return None
            
        return valid_preds[:, [5, 0, 1, 2, 3]]
        
# --- Verification Step ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Initialize the teacher and move it to the GPU/CPU
teacher_seg = SegmentationTeacher(device=DEVICE)

# 2. Create a dummy batch of RGB images
# Shape: [Batch Size, Channels, Height, Width]
dummy_rgb_input = torch.rand(2, 3, 480, 640).to(DEVICE)

# 3. Perform a forward pass
teacher_logits = teacher_seg(dummy_rgb_input)

# 4. Check the output shape
print("\n--- Teacher Integration Test ---")
print(f"Input Shape:  {dummy_rgb_input.shape}")
print(f"Output Shape: {teacher_logits.shape}")

# The output shape should be [Batch Size, 40, Height, Width]
# 40 is the number of classes in the NYUv2 dataset.
if teacher_logits.shape == (2, 10, 120, 160):
    print("✅ SUCCESS: The teacher model produced logits with the correct shape.")
else:
    print("❌ FAILURE: The output shape is incorrect. Please review the code.")


# --- Custom Collate Function ---
def hexapod_collate(batch):
    """ Custom collate to handle variable numbers of detection boxes per image. """
    keys = batch[0].keys()
    collated = {k: [d[k] for d in batch] for k in keys}
    
    # Stack tensors that have a fixed size
    for k in ['left', 'right', 'teacher', 'disp', 'seg']:
        collated[k] = torch.stack(collated[k])
    # 'det' remains a list of tensors
    return collated

# The indices we selected
NEIGHBORHOOD_IDX = 34795 
YOLO_IDX = 128923

# Function to pull and prepare a sample
def prepare_static_sample(dataset, idx):
    # Check if the index is in the training set or needs to be pulled from the base
    base_ds = dataset.dataset if hasattr(dataset, 'dataset') else dataset
    item = base_ds[idx]
    return {k: v.unsqueeze(0).to(DEVICE) if isinstance(v, torch.Tensor) else v 
            for k, v in item.items()}

def save_checkpoint(model, optimizer, scheduler, epoch, loss, filename):
    """
    Saves a checkpoint containing the model, optimizer, and scheduler states.
    """
    # Assumes SAVE_DIR is a globally defined config variable
    path = os.path.join(CONFIG['save_dir'], filename)
    
    checkpoint_data = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'loss': loss,
    }
    
    torch.save(checkpoint_data, path)
    print(f"💾 Checkpoint saved: {path}")





# --- Data Setup ---
DATA_ROOTS = {
    'coco':   '../datasets/coco',
    'tartan': '../datasets/TartanAir'
}

# 1. YOLO-Teacher laden
yolo_teacher = YoloTeacher(device=DEVICE)

# 2. Dataset mit Teacher-Option erstellen
# Wir geben dem Dataset Zugriff auf die Teacher-Logik
full_dataset = RealFusedDataset(
    roots=DATA_ROOTS,
    img_size=(CONFIG['img_height'], CONFIG['img_width']), # Hier fehlte die schließende Klammer )
    mode='train', 
    use_teacher=True,
    teacher_instance=yolo_teacher
)

train_size = int(0.95 * len(full_dataset))
train_ds, val_ds = random_split(full_dataset, [train_size, len(full_dataset) - train_size])

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True, collate_fn=hexapod_collate)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True, collate_fn=hexapod_collate)

# Save this for the whole training run
static_batches = {
    'yolo': prepare_static_sample(train_ds, YOLO_IDX),
    'neighborhood': prepare_static_sample(train_ds, NEIGHBORHOOD_IDX)
}




# Initialisierung
criterion_base = FusedHexapodLoss(CONFIG)
criterion = UncertaintyWeighting(criterion_base).to(DEVICE)

# 1. Parameter trennen
# Wir nehmen NUR die Parameter, die gradienten benötigen (also NICHT Backbone)
head_params = [p for p in model.parameters() if p.requires_grad]
criterion_params = list(criterion.parameters())

# 2. Optimizer Konfiguration
# Backbone fliegt raus -> Spart Rechenzeit im Optimizer Step
optimizer = torch.optim.AdamW([
    {'params': head_params,      'lr': 3e-3},  # Aggressiv: Die Heads müssen sich anpassen!
    {'params': criterion_params, 'lr': 1e-4}   # Loss-Gewichte schnell lernen
], weight_decay=1e-4) # Kleiner Decay, um Lernen nicht abzuwürgen

# 3. Scheduler anpassen
steps_per_epoch = len(train_loader) // CONFIG['ACCUMULATION_STEPS']

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[3e-3, 5e-3], # Nur noch 2 Gruppen!
    epochs=CONFIG['num_epochs'],
    steps_per_epoch=steps_per_epoch,
    pct_start=0.1,  # 10% Warmup
    div_factor=25,
    final_div_factor=1e4
)


#scaler = GradScaler()
# Ersetze: scaler = torch.cuda.amp.GradScaler()
# Durch diesen Block:

class DummyScaler:
    def __init__(self):
        pass
        
    def scale(self, loss):
        # Gibt den Loss einfach unverändert zurück
        return loss
        
    def step(self, optimizer):
        # Führt den normalen Optimizer-Step aus
        optimizer.step()
        
    def update(self):
        # Macht nichts (kein Skalierungsfaktor zu aktualisieren)
        pass
        
    def unscale_(self, optimizer):
        # Macht nichts (Werte sind schon unskaliert)
        pass

# Initialisierung
scaler = DummyScaler()
print("⚠️ Using DummyScaler for FP32 Training (No Mixed Precision)")

def set_frozen_state(model, criterion):
    """
    Konfiguriert das Modell für Phase 1 (Frozen Backbone).
    Aktiviert Gradienten nur für die Heads und die Loss-Gewichte.
    """
    # 1. Alles einfrieren (Basis-Zustand: Backbone & alles andere ist dicht)
    model.requires_grad_(False)
    
    # 2. Heads explizit auftauen
    # Da 'stereo_head' jetzt auch die Reducer und Refinement-Stages enthält,
    # reicht dieser eine Aufruf, um den gesamten Stereo-Zweig zu aktivieren.
    model.stereo_head.requires_grad_(True)
    model.yolo_head.requires_grad_(True)
    model.seg_head.requires_grad_(True)
    
    # 3. Loss-Gewichte auftauen (Uncertainty Weighting Parameter)
    # WICHTIG: Das Criterion muss auch trainierbar sein!
    criterion.requires_grad_(True)
    
    # --- Check & Print Stats ---
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    
    print(f"\n❄️  Model State Configured:")
    print(f"   -> Backbone:     FROZEN 🧊")
    print(f"   -> Stereo Head:  TRAINABLE 🔥 (inkl. Reducer & Refinement)")
    print(f"   -> YOLO Head:    TRAINABLE 🔥")
    print(f"   -> Seg Head:     TRAINABLE 🔥")
    print(f"   -> Loss Weights: TRAINABLE 🔥")
    print(f"   📊 Params: {trainable_params/1e6:.2f}M Trainable / {total_params/1e6:.2f}M Total")

# Aufruf im Skript:
# set_frozen_state(model, criterion)



model.train()

def train_one_epoch(epoch_idx):
    model.train()
    model.float()
    # if backbone_frozen: model.backbone.eval() # verhindert BN-running-stats drift im Backbone
    criterion.train()
    running_loss = 0.0
    batches_processed = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch_idx+1}/{CONFIG['num_epochs']}")

    optimizer.zero_grad()
    
    for i, batch in enumerate(pbar):
        global_step = epoch_idx * len(train_loader) + i
        
        l, r, t_img = batch['left'].to(DEVICE), batch['right'].to(DEVICE), batch['teacher'].to(DEVICE)
        targets = {
            'disp': batch['disp'].to(DEVICE), 
            'seg': batch['seg'].to(DEVICE), 
            'det': [t.to(DEVICE) for t in batch['det']]
        }

        # 1. Lehrer-Vorhersage
        with torch.no_grad():
            teacher_preds = teacher_seg(t_img)

        # 2. Forward Pass mit Mixed Precision
        #with torch.amp.autocast('cuda'):
        with nullcontext():
            preds = model(l, r)
            
            # Nutzt jetzt UncertaintyWeighting(FusedHexapodLoss)
            loss, logs = criterion(preds, targets, teacher_preds, left_img=l, step=global_step)

            
            # Skalierung für Batch Akkumulation
            distorted_loss = loss / CONFIG['ACCUMULATION_STEPS']

        # 3. Sicherheitscheck (NaN/Inf)
        if not torch.isfinite(loss):
            print(f"⚠️ Non-finite loss detected at step {i}!")
            print(f"   Breakdown: {logs}")
            optimizer.zero_grad()
            continue 

        # 4. Backward Pass (Skaliert)
        scaler.scale(distorted_loss).backward()
        running_loss += loss.item()
        batches_processed += 1

        # 5. Optimizer Step (nach Akkumulations-Phasen)
        if (i + 1) % CONFIG['ACCUMULATION_STEPS'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step() 
        # 6. Detailliertes Logging (alle 50 Batches)
        if i % 50 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            writer.add_scalar('Loss/Total_Weighted', loss.item(), global_step)
            writer.add_scalar('Params/LearningRate', current_lr, global_step)
            
            # --- INTERNE TRENDS & GEWICHTE LOGGEN ---
            for k, v in logs.items():
                if k.startswith('weight_'):
                    # Zeigt wie wichtig das Modell den Task gerade einschätzt
                    writer.add_scalar(f'Dynamic_Weights/{k}', v, global_step)
                else:
                    # Der rohe, ungewichtete Loss für echtes Performance-Tracking
                    writer.add_scalar(f'Loss_Raw/{k}', v, global_step)

        # 7. Dashboard Visualisierung
        if i % 500 == 0:
            visualize_hexapod_output(step=global_step, writer=writer)
            model.train()

        # TQDM Statuszeile mit dynamischen Gewichten
        pbar.set_postfix({
            'L': f"{loss.item():.2f}", 
            # W_Y zeigt das aktuelle Gewicht für YOLO (Precision)
            'W_Y': f"{logs.get('weight_yolo', 0):.2f}",
            'W_S': f"{logs.get('weight_stereo', 0):.2f}",
            'W_Seg': f"{logs.get('weight_seg', 0):.2f}"
        })
    # 1. Calculate average loss for the epoch
    avg_loss = running_loss / batches_processed if batches_processed > 0 else 0.0
    save_checkpoint(
        model=model, 
        optimizer=optimizer, 
        scheduler=scheduler, 
        epoch=epoch_idx, 
        loss=avg_loss, # Use the calculated average loss
        filename=f"checkpoint_epoch_{epoch_idx}.pth"
    )    
    return avg_loss

print("✅ train_one_epoch mit Akkumulation und interner Trend-Analyse aktualisiert.")
    
# --- Main Execution ---


# Configuration
RESUME_PATH = "./checkpoints/checkpoint_epoch_3.pth" # Or None to start fresh
# RESUME_PATH = "./checkpoints/model_epoch_15.pth" 

start_epoch = 0

if RESUME_PATH and start_epoch!=0:
    print(f"🔄 Resuming training from: {RESUME_PATH}")
    checkpoint = torch.load(RESUME_PATH, map_location=DEVICE)
    
    # 1. Load Model Weights
    model.load_state_dict(checkpoint['model_state'])
    
    # 2. Load Optimizer State (Critical for Adam/AdamW)
    if 'optimizer_state' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        
    # 3. Load Scalar (if using AMP)
    if 'scaler' in checkpoint and 'scaler' in locals():
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        
    # 4. Set Start Epoch
    start_epoch = checkpoint['epoch'] + 1
    print(f"✅ Successfully loaded. Starting at Epoch {start_epoch}")

else:
    print("✨ Starting fresh training.")    
# --- Main Execution ---
print("🚀 Starting training...")

# Define how long to freeze
FREEZE_EPOCHS = 5 

set_frozen_state(model, criterion=criterion)



    




--- Teacher Integration Test ---
Input Shape:  torch.Size([2, 3, 480, 640])
Output Shape: torch.Size([2, 10, 120, 160])
✅ SUCCESS: The teacher model produced logits with the correct shape.
   Scanning FlyingThings3D...
   -> Found 21818 FT3D pairs.
   Scanning TartanAir...
   -> Found 38603 TartanAir samples.
   Scanning COCO 2017...
loading annotations into memory...
Done (t=10.63s)
creating index...
index created!
   -> Found 117266 COCO samples.
📊 Total Samples Found: 177687
⚠️ Using DummyScaler for FP32 Training (No Mixed Precision)
✅ train_one_epoch mit Akkumulation und interner Trend-Analyse aktualisiert.
✨ Starting fresh training.
🚀 Starting training...

❄️  Model State Configured:
   -> Backbone:     FROZEN 🧊
   -> Stereo Head:  TRAINABLE 🔥 (inkl. Reducer & Refinement)
   -> YOLO Head:    TRAINABLE 🔥
   -> Seg Head:     TRAINABLE 🔥
   -> Loss Weights: TRAINABLE 🔥
   📊 Params: 7.60M Trainable / 10.57M Total


In [ ]:
for epoch in range(start_epoch, CONFIG['num_epochs']):
    # --- ÜBERGANG: BACKBONE AUFTAUEN (z.B. Start Epoche 5) ---
    if epoch == FREEZE_EPOCHS:
        print("\n🧊 -> 🔥 PHASE 2: Unfreezing Backbone & Resetting Scheduler")
        
        # A) Modell komplett für Gradienten öffnen
        model.requires_grad_(True)
            
        # B) VRAM Schutz
        torch.cuda.empty_cache()

        # C) FEINGRANULARE PARAMETER-GRUPPEN (Der "Balanced" Ansatz)
        # Wir trennen die Heads, um individuelle LRs zu ermöglichen
        backbone_params = list(model.backbone.parameters())
        yolo_params     = list(model.yolo_head.parameters())
        stereo_params   = list(model.stereo_head.parameters())
        seg_params      = list(model.seg_head.parameters())
        criterion_params = list(criterion.parameters())

        # D) BALANCED OPTIMIZER (Differential Learning Rates)
        # Die LRs sind so gewählt, dass Stereo die Details lernt und YOLO stabil bleibt
        base_lr = 1e-4
        optimizer = torch.optim.AdamW([
            {'params': backbone_params,  'lr': base_lr * 0.1, 'weight_decay': 1e-5}, # Sehr vorsichtig
            {'params': yolo_params,      'lr': base_lr * 0.5}, # YOLO ist schnell (dank Teacher)
            {'params': stereo_params,    'lr': base_lr * 2.0}, # Stereo braucht mehr "Druck" für Pixel-Details
            {'params': seg_params,       'lr': base_lr * 1.0}, # Seg moderat
            {'params': criterion_params, 'lr': base_lr * 1.0}  # Uncertainty Weights
        ], weight_decay=1e-4)

        # E) NEUER SCHEDULER (Mit passenden Max-LRs für jede Gruppe)
        remaining_epochs = CONFIG['num_epochs'] - epoch
        steps_per_epoch = len(train_loader) // CONFIG['ACCUMULATION_STEPS']
        
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=[
                base_lr * 0.1, # Backbone
                base_lr * 0.5, # YOLO
                base_lr * 2.0, # Stereo
                base_lr * 1.0, # Seg
                base_lr * 1.0  # Criterion
            ],
            epochs=remaining_epochs,
            steps_per_epoch=steps_per_epoch,
            pct_start=0.2,            # Kürzere Warmup-Phase, da wir nicht bei Null anfangen
            div_factor=10,            # Startet näher an der Max-LR
            final_div_factor=1e4
        )
        print(f"✅ Balanced Optimizer & Scheduler reset for Phase 2!")
        print(f"   LR-Verteilung: YOLO={base_lr*0.5:.1e}, Stereo={base_lr*2.0:.1e}, Backbone={base_lr*0.1:.1e}")

    # --- Training Steps ---
    epoch_loss = train_one_epoch(epoch)
    print(f'Epoch {epoch} done. Average epoch loss = {epoch_loss:.4f}')
        

print("🏁 Training complete.")

In [10]:
%abort

UsageError: Line magic function `%abort` not found.


In [38]:
import math
from tqdm.auto import tqdm

def reset_model_heads(model):
    print("re-initializing heads to clean slate...")

    def weight_init(m):
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
            # Kaiming He Initialisierung für ReLU-Aktivierungen
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

    # 1. Stereo Head resetten
    model.stereo_head.apply(weight_init)
    # Wichtig: Den Refinement-Bias am Ende wieder auf 0 setzen für Stabilität
    if hasattr(model.stereo_head.stereo_refine_s1.net[-1], 'weight'):
        nn.init.constant_(model.stereo_head.stereo_refine_s1.net[-1].weight, 0)
        nn.init.constant_(model.stereo_head.stereo_refine_s1.net[-1].bias, 0)

    # 2. Segmentation Head resetten
    model.seg_head.apply(weight_init)

    # 3. YOLO Head resetten
    model.yolo_head.apply(weight_init)
    
    # 4. Spezielle YOLO-Initialisierung (dein Prior-Bias-Tool)
    # Das ist wichtig, damit die Confidence nicht am Anfang explodiert!
    initialize_yolo_head(model)

    print("✅ Heads reset. Backbone preserved.")

# Aufruf:
# reset_model_heads(model)


def run_isolation_debug(task_type, num_reps=100, lr=1e-4, train_backbone=False):
    mode_str = "BACKBONE_UNFROZEN" if train_backbone else "BACKBONE_FROZEN"
    print(f"🛠️ Isoliertes Training: {task_type.upper()} | {num_reps} Reps | {mode_str}")

    # 1. Modell-Zustand vorbereiten
    model.train()
    # Zuerst alles einfrieren
    model.requires_grad_(False)
    
    # Nur den Ziel-Head und optional den Backbone auftauen
    if task_type == 'stereo':
        model.stereo_head.requires_grad_(True)
    elif task_type == 'seg':
        model.seg_head.requires_grad_(True)
    elif task_type == 'yolo':
        model.yolo_head.requires_grad_(True)
        
    if train_backbone:
        model.backbone.requires_grad_(True)
    else:
        model.backbone.eval() # BN-Stats einfrieren

    # Optimizer nur für die aktiven Parameter
    active_params = [p for p in model.parameters() if p.requires_grad]
    debug_optimizer = torch.optim.AdamW(active_params, lr=lr)

    # Daten holen
    batch = static_batches['yolo'] if task_type == 'yolo' else static_batches['neighborhood']
    l, r = batch['left'].to(DEVICE), batch['right'].to(DEVICE)
    t_img = batch['teacher'].to(DEVICE)
    targets = {
        'disp': batch['disp'].to(DEVICE),
        'seg':  batch['seg'].to(DEVICE),
        'det':  [t.to(DEVICE) for t in batch['det']]
    }

    pbar = tqdm(range(num_reps), desc=f"Debug {task_type}")
    for i in pbar:
        debug_optimizer.zero_grad()
        
        with torch.no_grad():
            teacher_preds = teacher_seg(t_img)
            
        # Forward Pass
        d_raw, s_raw, y_raw = model(l, r)
        
        # --- WASSERDICHTE ISOLATION ---
        # Wir detachen alles, was NICHT zum aktuellen Task gehört.
        # So fließen Gradienten GARANTIERT nur vom Ziel-Task in den Backbone.
        d_iso = d_raw if task_type == 'stereo' else d_raw.detach()
        s_iso = s_raw if task_type == 'seg' else s_raw.detach()
        y_iso = y_raw if task_type == 'yolo' else [y.detach() for y in y_raw]
        
        preds = (d_iso, s_iso, y_iso)
        
        # Loss-Berechnung (Nutzt die aktuellen log_vars, aber nur für den Ziel-Task)
        loss, logs = criterion(preds, targets, teacher_preds, left_img=l)
        
        if not math.isfinite(loss.item()):
            print(f"❌ Abbruch: NaN")
            break
            
        loss.backward()
        torch.nn.utils.clip_grad_norm_(active_params, 1.0)
        debug_optimizer.step()
        
        # Logging
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        writer.add_scalar(f'Debug_Isolation/{task_type}_{mode_str}_Total', loss.item(), i)
        
        # Einzel-Komponenten loggen
        for k in logs:
            if task_type in k or (task_type == 'seg' and 'kd' in k):
                writer.add_scalar(f'Debug_Isolation_Raw/{task_type}_{k}', logs[k], i)

        if i % (num_reps // 50) == 0 or i == num_reps - 1:
            visualize_hexapod_output(step=i, writer=writer)

    print(f"✅ Debug-Lauf beendet.")

In [45]:
reset_model_heads(model)
run_isolation_debug('yolo', num_reps=5000, lr=1e-3, train_backbone=False)

re-initializing heads to clean slate...
✅ Initialized 3 YOLO Heads with Prior Bias.
✅ Heads reset. Backbone preserved.
🛠️ Isoliertes Training: YOLO | 5000 Reps | BACKBONE_FROZEN


Debug yolo: 100%|███████████████████████████████████████████| 5000/5000 [11:34<00:00,  7.20it/s, loss=60.5759]

✅ Debug-Lauf beendet.


In [ ]:
reset_model_heads(model)
run_isolation_debug('yolo', num_reps=150, lr=1e-4, train_backbone=True)

In [44]:
reset_model_heads(model)
run_isolation_debug('stereo', num_reps=5000, lr=2e-4, train_backbone=False)

re-initializing heads to clean slate...
✅ Initialized 3 YOLO Heads with Prior Bias.
✅ Heads reset. Backbone preserved.
🛠️ Isoliertes Training: STEREO | 5000 Reps | BACKBONE_FROZEN


Debug stereo: 100%|█████████████████████████████████████████| 5000/5000 [11:58<00:00,  6.96it/s, loss=51.5532]

✅ Debug-Lauf beendet.


In [ ]:
reset_model_heads(model)
run_isolation_debug('stereo', num_reps=50, lr=2e-5, train_backbone=True)

In [ ]:
reset_model_heads(model)
run_isolation_debug('seg', num_reps=5000, lr=2e-4, train_backbone=False)

In [ ]:
reset_model_heads(model)
run_isolation_debug('seg', num_reps=80, lr=5e-5, train_backbone=True)

In [55]:
def reset_heads_and_criterion(model, criterion):
    print("🧹 Resetting all heads and uncertainty weights to clean slate...")
    
    def init_weights(m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

    # 1. Heads resetten
    model.stereo_head.apply(init_weights)
    model.seg_head.apply(init_weights)
    model.yolo_head.apply(init_weights)
    
    # 2. Stereo Refinement auf Zero-Residual setzen
    nn.init.constant_(model.stereo_head.stereo_refine_s1.net[-1].weight, 0)
    nn.init.constant_(model.stereo_head.stereo_refine_s1.net[-1].bias, 0)
    
    # 3. YOLO Bias Spezial-Init (Prior 0.01)
    initialize_yolo_head(model) 

    # 4. CRITICAL: Uncertainty Weights zurücksetzen
    # Wir setzen alle auf 0.0 (entspricht einem Startgewicht von 1.0)
    with torch.no_grad():
        criterion.log_vars.fill_(0.0)
    
    print("✅ System is fresh. No bias, no stale weights.")

import math
from tqdm.auto import tqdm

def run_combined_debug(num_reps=100, lr=1e-4, train_backbone=False):
    mode_str = "BACKBONE_UNFROZEN" if train_backbone else "BACKBONE_FROZEN"
    print(f"🚀 Combined Dual-Test: YOLO & NEIGHBORHOOD | {num_reps} Reps | {mode_str}")

    model.train()
    if train_backbone:
        model.requires_grad_(True)
    else:
        model.requires_grad_(False)
        model.stereo_head.requires_grad_(True)
        model.seg_head.requires_grad_(True)
        model.yolo_head.requires_grad_(True)
        model.backbone.eval()

    active_params = [p for p in model.parameters() if p.requires_grad]
    debug_optimizer = torch.optim.AdamW(active_params, lr=lr)

    # Wir definieren die beiden Batches, die wir parallel optimieren
    test_batches = [
        ('yolo', static_batches['yolo']),
        ('neighborhood', static_batches['neighborhood'])
    ]

    pbar = tqdm(range(num_reps), desc="Dual-Batch Optimization")
    for i in pbar:
        debug_optimizer.zero_grad()
        
        total_step_loss = 0
        step_logs = {}

        # Wir prozessieren beide Batches hintereinander, bevor wir den Optimizer-Schritt machen
        for name, batch in test_batches:
            l, r = batch['left'].to(DEVICE), batch['right'].to(DEVICE)
            t_img = batch['teacher'].to(DEVICE)
            targets = {
                'disp': batch['disp'].to(DEVICE),
                'seg':  batch['seg'].to(DEVICE),
                'det':  [t.to(DEVICE) for t in batch['det']]
            }

            with torch.no_grad():
                teacher_preds = teacher_seg(t_img)
                
            preds = model(l, r)
            
            # Weighted Loss berechnen
            loss, logs = criterion(preds, targets, teacher_preds, left_img=l)
            
            # Gradienten akkumulieren (skaliert auf Anzahl der Batches)
            (loss / len(test_batches)).backward()
            
            total_step_loss += loss.item()
            # Logs für Tensorboard sammeln (wir nehmen den Durchschnitt oder summieren)
            for k, v in logs.items():
                step_logs[k] = step_logs.get(k, 0) + v

        torch.nn.utils.clip_grad_norm_(active_params, 1.0)
        debug_optimizer.step()
        
        # Logging
        avg_loss = total_step_loss / len(test_batches)
        pbar.set_postfix({'loss': f"{avg_loss:.3f}"})
        
        writer.add_scalar(f'Debug_Combined/Total_{mode_str}', avg_loss, i)
        for k, v in step_logs.items():
            val = v / len(test_batches)
            if not k.startswith('weight_'):
                writer.add_scalar(f'Debug_Combined_Raw/{mode_str}_{k}', val, i)
            else:
                writer.add_scalar(f'Debug_Combined_Weights/{mode_str}_{k}', val, i)

        # Visualisierung: Da visualize_hexapod_output() intern beide Batches nutzt,
        # sehen wir hier den Fortschritt auf beiden Bild-Paaren gleichzeitig!
        if i % max(1, (num_reps // 50)) == 0 or i == num_reps - 1:
            visualize_hexapod_output(step=i, writer=writer)

    print(f"✅ Combined Dual-Test beendet.")

In [56]:
# Hohe LR für die Heads
reset_heads_and_criterion(model, criterion)
run_combined_debug(num_reps=5000, lr=1e-3, train_backbone=False)

🧹 Resetting all heads and uncertainty weights to clean slate...
✅ Initialized 3 YOLO Heads with Prior Bias.
✅ System is fresh. No bias, no stale weights.
🚀 Combined Dual-Test: YOLO & NEIGHBORHOOD | 5000 Reps | BACKBONE_FROZEN


Dual-Batch Optimization:  96%|██████████████████████████████▌ | 4778/5000 [22:59<01:04,  3.46it/s, loss=0.385]


KeyboardInterrupt: 

In [ ]:
# Niedrigere LR, damit der Backbone nicht explodiert
reset_heads_and_criterion(model, criterion)
run_combined_debug(num_reps=150, lr=1e-4, train_backbone=True)